# Week 06 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 데이터와 자동 확인 도우미

자동 확인은 임시 폴더(`tempfile`)에 테스트용 파일을 만들어 확인하므로 여러분의 폴더를 어지럽히지 않습니다.

In [ ]:
import csv
import tempfile
import urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

STUDENTS = get_data("students.csv")
TMP = Path(tempfile.mkdtemp())


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")


def write_text(name, text):
    """테스트용 임시 파일을 만들고 경로를 돌려준다."""
    p = TMP / name
    p.write_text(text, encoding="utf-8")
    return str(p)

---
## 문제 1. 안전한 정수 변환 (2점)

`safe_int(text, default=None, low=None, high=None)`

- 앞뒤 공백을 지우고 정수로 바꿉니다.
- 정수로 바꿀 수 없으면 `default`를 돌려줍니다. (`"3.5"`, `"abc"`, `""` 등)
- `low`나 `high`가 주어졌는데 범위를 벗어나면 `default`를 돌려줍니다. (경계값 포함)
- **`try/except ValueError`** 를 사용하고, `except:`(무엇이든 잡기)는 쓰지 마세요.

In [ ]:
def safe_int(text, default=None, low=None, high=None):
    """정수 변환에 실패하거나 범위를 벗어나면 default를 돌려줍니다."""
    try:
        value = int(text.strip())
    except ValueError:
        return default
    if low is not None and value < low:
        return default
    if high is not None and value > high:
        return default
    return value

In [ ]:
check("' 42 ' → 42", lambda: safe_int(" 42 ") == 42)
check("'abc' → None", lambda: safe_int("abc") is None)
check("'' → default 0", lambda: safe_int("", default=0) == 0)
check("'3.5' → default", lambda: safe_int("3.5", default=-1) == -1)
check("'-7' → -7", lambda: safe_int("-7") == -7)
check("범위 안 경계 100", lambda: safe_int("100", low=0, high=100) == 100)
check("범위 밖 101 → default", lambda: safe_int("101", default=0, low=0, high=100) == 0)
check("low만 지정", lambda: safe_int("-1", default="X", low=0) == "X")

---
## 문제 2. 메모 파일 다루기 (2점)

세 함수를 완성하세요. 모든 파일은 `encoding="utf-8"`로 다룹니다.

1. `save_lines(path, lines)`: 문자열 리스트를 한 줄에 하나씩 저장합니다. **기존 내용은 지웁니다.**
2. `append_line(path, line)`: 파일 끝에 한 줄을 덧붙입니다. 파일이 없으면 새로 만듭니다.
3. `count_lines(path)`: **내용이 있는(공백만 있는 줄 제외)** 줄 수를 돌려줍니다. 파일이 없으면 `0`을 돌려줍니다. (`FileNotFoundError` 처리)

In [ ]:
def save_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")


def append_line(path, line):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def count_lines(path):
    try:
        with open(path, encoding="utf-8") as f:
            return sum(1 for line in f if line.strip())
    except FileNotFoundError:
        return 0

In [ ]:
memo = str(TMP / "memo.txt")
check("저장 후 3줄", lambda: (save_lines(memo, ["월", "화", "수"]), count_lines(memo))[1] == 3)
check("이어 쓰기 후 4줄", lambda: (append_line(memo, "목"), count_lines(memo))[1] == 4)
check("다시 저장하면 덮어씀", lambda: (save_lines(memo, ["새 내용"]), count_lines(memo))[1] == 1)
check("내용이 정확함", lambda: (save_lines(memo, ["가", "나"]), Path(memo).read_text(encoding="utf-8"))[1] == "가\n나\n")
check("없는 파일 → 0", lambda: count_lines(str(TMP / "없는파일.txt")) == 0)
check("빈 줄·공백 줄 제외", lambda: count_lines(write_text("blank.txt", "a\n\n   \nb\n")) == 2)
check("없는 파일에 append → 새로 생성", lambda: (append_line(str(TMP / "new.txt"), "첫 줄"), count_lines(str(TMP / "new.txt")))[1] == 1)

---
## 문제 3. CSV 학과별 평균 (3점)

`dept_average(path, column)`은 CSV 파일에서 학과(`department`)별로 `column` 열의 평균을 구해 `{학과: 평균}` 딕셔너리로 돌려줍니다.

- `csv.DictReader`를 사용합니다.
- 해당 열이 **비어 있거나 숫자가 아닌 행은 건너뜁니다.** (문제 1의 `safe_int`를 활용하세요.)
- 평균은 소수 첫째 자리까지 반올림합니다.
- 유효한 값이 하나도 없는 학과는 결과에서 **뺍니다.**
- `column`이 파일에 없는 열이면 `KeyError`가 그대로 발생해도 됩니다.

In [ ]:
def dept_average(path, column):
    """학과별 column 평균 {학과: 평균}."""
    groups = {}
    with open(path, encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            value = safe_int(row[column])
            if value is None:
                continue
            groups.setdefault(row["department"].strip(), []).append(value)
    return {dept: round(sum(v) / len(v), 1) for dept, v in groups.items()}

In [ ]:
check("students.csv final 평균", lambda: dept_average(STUDENTS, "final") ==
      {"간호학과": 71.7, "경영학과": 70.5, "전자공학과": 75.2, "정보보안학과": 74.0, "컴퓨터공학과": 73.6})
check("students.csv assignment (빈 칸 건너뜀)", lambda: dept_average(STUDENTS, "assignment")["정보보안학과"] == 82.3)

mini = write_text("mini.csv", "department,score\n가학과,80\n가학과,abc\n나학과,\n가학과, 90 \n다학과,70\n")
check("잘못된 값 건너뜀", lambda: dept_average(mini, "score") == {"가학과": 85.0, "다학과": 70.0})


def _raises_keyerror(p):
    try:
        dept_average(p, "없는열")
    except KeyError:
        return True
    return False

check("없는 열이면 KeyError", lambda: _raises_keyerror(mini))

---
## 문제 4. 검증 보고서 만들기 (AI 협업, 3점)

수강 신청 CSV를 검증해 **정상 행 목록**과 **오류 보고서 파일**을 만드는 `validate_enrollment(in_path, report_path)`를 작성하세요.

**입력 열**: `student_id`, `name`, `credits`

| 열 | 규칙 | 오류 이유(reason) 문자열 |
|---|---|---|
| `student_id` | 8자리 숫자 | `"student_id 형식"` |
| `student_id` | 앞에서 이미 나온 학번 | `"student_id 중복"` |
| `name` | 공백 제거 후 비어 있지 않음 | `"name 비어 있음"` |
| `credits` | 1~21 정수 | `"credits 범위"` |

- 한 행에 여러 문제가 있으면 **위 표의 순서대로 처음 발견한 하나만** 기록합니다.
- 줄 번호(`line`)는 헤더가 1번 줄, 첫 데이터가 2번 줄입니다.
- 반환값: 정상 행 리스트 `[{"student_id": str, "name": str, "credits": int}, ...]`
- `report_path`에 `line,student_id,reason` 헤더를 가진 CSV로 오류를 저장합니다. 오류가 없어도 헤더는 씁니다.
- 중복 판정은 **정상으로 통과한 학번**만 기준으로 합니다.

**AI 협업 과제**: AI의 첫 답에 `except:`나 `except Exception: pass`가 있었는지, 줄 번호가 맞는지 확인하고 기록하세요.

In [ ]:
def validate_enrollment(in_path, report_path):
    """정상 행 리스트를 돌려주고, 오류는 report_path CSV로 저장합니다."""
    valid, errors, seen = [], [], set()
    with open(in_path, encoding="utf-8", newline="") as f:
        for line_no, row in enumerate(csv.DictReader(f), start=2):
            sid = row["student_id"].strip()
            name = row["name"].strip()
            credits = safe_int(row["credits"], low=1, high=21)
            if not (len(sid) == 8 and sid.isdigit()):
                reason = "student_id 형식"
            elif sid in seen:
                reason = "student_id 중복"
            elif not name:
                reason = "name 비어 있음"
            elif credits is None:
                reason = "credits 범위"
            else:
                seen.add(sid)
                valid.append({"student_id": sid, "name": name, "credits": credits})
                continue
            errors.append({"line": line_no, "student_id": sid, "reason": reason})
    with open(report_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["line", "student_id", "reason"])
        writer.writeheader()
        writer.writerows(errors)
    return valid

In [ ]:
enroll = write_text("enroll.csv", "\n".join([
    "student_id,name,credits",
    "20261001,김민준,18",      # 2 정상
    "2026100,이서연,15",       # 3 형식
    "20261002, ,12",           # 4 이름
    "20261003,박지호,24",      # 5 학점 범위
    "20261001,김민준,18",      # 6 중복
    "20261004,최하은, 21 ",    # 7 정상 (경계, 공백)
    "abcdefgh,정도윤,abc",     # 8 형식 (먼저 발견)
    "20261003,박지호,15",      # 9 정상 (5번 줄은 오류였으므로 중복 아님)
]) + "\n")
report = str(TMP / "report.csv")


def read_report():
    with open(report, encoding="utf-8", newline="") as f:
        return [(int(r["line"]), r["reason"]) for r in csv.DictReader(f)]


check("정상 행 3개", lambda: [r["student_id"] for r in validate_enrollment(enroll, report)] == ["20261001", "20261004", "20261003"])
check("credits는 int로 변환", lambda: validate_enrollment(enroll, report)[1]["credits"] == 21)
check("보고서 내용과 줄 번호", lambda: (validate_enrollment(enroll, report), read_report())[1] ==
      [(3, "student_id 형식"), (4, "name 비어 있음"), (5, "credits 범위"), (6, "student_id 중복"), (8, "student_id 형식")])
ok_file = write_text("ok.csv", "student_id,name,credits\n20260001,가,1\n")
check("오류 없어도 헤더는 씀", lambda: (validate_enrollment(ok_file, report), Path(report).read_text(encoding="utf-8").strip())[1] == "line,student_id,reason")

### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) ChatGPT |
| AI 첫 답의 예외 처리 | 행 전체를 `try: ... except Exception: continue`로 감싸 오류 이유가 기록되지 않았음 |
| 실패한 항목 | 줄 번호를 `enumerate(reader)`(0부터)로 세어 1씩 어긋남, 오류가 난 5번 줄 학번을 `seen`에 넣어 9번 줄을 중복으로 처리 |
| 직접 고친 부분 | `enumerate(..., start=2)`, 정상 통과 후에만 `seen.add`, 규칙 순서대로 `if-elif`로 이유 결정 |

---
## 성찰 (필수)

1. "건너뛰어야 할 오류"와 "드러나야 할 오류(버그)"를 어떻게 구분했나요?
2. 오류 보고서에 줄 번호와 이유를 남기면 누구에게 어떤 도움이 될까요?
3. CSV 값이 모두 문자열이라는 점 때문에 겪은 문제가 있었나요?

(학생 작성 영역)